# Data Engineering — Moduł 6: Modelowanie danych i architektura hurtowni

Moduły 4-5 nauczyły nas BUDOWAĆ pojedyncze modele dbt. Ten moduł uczy, jak zorganizować
DZIESIĄTKI takich modeli w spójną, przewidywalną strukturę, którą analitycy (i BI) mogą
efektywnie odpytywać -- **modelowanie wymiarowe** (Kimball), fakty i wymiary, schemat
gwiazdy. Budujemy praktyczną hurtownię nad naszym znanym zestawem danych sklepu
internetowego.

## Spis treści
1. [Kimball kontra Inmon](#sec1)
2. [Fakty i wymiary](#sec2)
3. [Schemat gwiazdy kontra płatek śniegu](#sec3)
4. [Typy tabel faktów](#sec4)
5. [Budujemy schemat gwiazdy](#sec5)
6. [Podsumowanie i ćwiczenia](#sec6)


Generujemy nasz znany zestaw danych sklepu internetowego i ładujemy go do **DuckDB** (lekka, plikowa baza analityczna) jako schemat `raw` -- symulując surowe dane, które w prawdziwym projekcie załadowałby zewnętrzny proces ingestii (Moduł 1). Budujemy też szkielet projektu dbt (`dbt_project.yml`, `profiles.yml`, `sources.yml`) wskazujący na te dane.

In [ ]:
import os
import subprocess
import duckdb
import numpy as np
import pandas as pd

# --- 1. Generujemy surowe dane (ten sam generator co w kursie Databricks/PySpark) ---
rng = np.random.default_rng(7)

n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=[0.35, 0.45, 0.20]),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}
kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2) for k in kategorie_produktow],
})

n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values
prawdop_anulowania = np.where(segment_zamowien == "Nowy", 0.22, np.where(segment_zamowien == "Standardowy", 0.10, 0.04))
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(anulowane == 1, "Anulowane", rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]))
data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)
zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]
for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id, "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id), "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]), "rabat": float(rabat),
        })
        pozycja_id += 1
pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- 2. Ladujemy surowe dane do DuckDB, schemat "raw" (jakby zaladowal je zewnetrzny proces ingestii) ---
DBT_DIR = os.path.abspath("dbt_kurs")
for _poddir in ("models", "tests", "macros", "seeds", "snapshots"):
    os.makedirs(f"{DBT_DIR}/{_poddir}", exist_ok=True)
DUCKDB_PATH = f"{DBT_DIR}/kurs_de.duckdb"

con = duckdb.connect(DUCKDB_PATH)
con.execute("CREATE SCHEMA IF NOT EXISTS raw")
con.execute("CREATE OR REPLACE TABLE raw.klienci AS SELECT * FROM klienci_pd")
con.execute("CREATE OR REPLACE TABLE raw.produkty AS SELECT * FROM produkty_pd")
con.execute("CREATE OR REPLACE TABLE raw.zamowienia AS SELECT * FROM zamowienia_pd")
con.execute("CREATE OR REPLACE TABLE raw.pozycje_zamowien AS SELECT * FROM pozycje_zamowien_pd")
con.close()

# --- 3. Szkielet projektu dbt ---
with open(f"{DBT_DIR}/dbt_project.yml", "w") as f:
    f.write(
        "name: kurs_de\n"
        "version: '1.0'\n"
        "profile: kurs_de\n"
        "model-paths: [\"models\"]\n"
        "seed-paths: [\"seeds\"]\n"
        "snapshot-paths: [\"snapshots\"]\n"
    )

with open(f"{DBT_DIR}/profiles.yml", "w") as f:
    f.write(
        "kurs_de:\n"
        "  target: dev\n"
        "  outputs:\n"
        "    dev:\n"
        "      type: duckdb\n"
        f"      path: {DUCKDB_PATH}\n"
    )

with open(f"{DBT_DIR}/models/sources.yml", "w") as f:
    f.write(
        "version: 2\n"
        "sources:\n"
        "  - name: raw_dane\n"
        "    schema: raw\n"
        "    tables:\n"
        "      - name: klienci\n"
        "      - name: produkty\n"
        "      - name: zamowienia\n"
        "      - name: pozycje_zamowien\n"
    )

env = os.environ.copy()
env["DBT_PROFILES_DIR"] = DBT_DIR


def dbt(*args, check=True):
    """Uruchamia komende `dbt ...` w naszym projekcie i wypisuje wynik."""
    wynik = subprocess.run(
        ["dbt", *args, "--project-dir", DBT_DIR],
        env=env, capture_output=True, text=True,
    )
    print(wynik.stdout[-4000:])
    if wynik.returncode != 0:
        print(wynik.stderr[-2000:])
        if check:
            raise RuntimeError(f"dbt {' '.join(args)} zakonczyl sie kodem {wynik.returncode}")
    return wynik


print(f"Zaladowano surowe dane do {DUCKDB_PATH} (schemat raw).")
print(f"Projekt dbt: {DBT_DIR}")


<a id="sec1"></a>
## 1. Kimball kontra Inmon

Dwie klasyczne, konkurencyjne filozofie budowania hurtowni danych, obie z lat 90., wciąż
aktualne:

- **Ralph Kimball (bottom-up)** -- buduj hurtownię PO KAWAŁKU, dziedzina po dziedzinie
  (najpierw "Sprzedaż", potem "Marketing", potem "HR"...), każda jako osobny
  **schemat gwiazdy** (sekcja 3). Szybszy start, wartość biznesowa widoczna od razu.
  Ryzyko: bez dyscypliny różne dziedziny mogą zdefiniować te same pojęcia (np.
  "klient") niespójnie.
- **Bill Inmon (top-down)** -- najpierw zbuduj JEDEN, w pełni znormalizowany model
  całego przedsiębiorstwa (*Enterprise Data Warehouse*, forma zbliżona do 3NF z Modułu 8
  kursu SQL), a dopiero z niego wydzielaj mniejsze, zdenormalizowane **data marty** dla
  poszczególnych zespołów. Więcej pracy na start, ale gwarantowana spójność definicji w
  całej firmie.

**W praktyce większość nowoczesnych zespołów (i ten kurs) idzie w stronę Kimballa** --
szybciej dowozi wartość, a dbt (z `ref()` budującym wspólne, reużywalne modele
`staging`) łagodzi główne ryzyko niespójności.

> 🏭 **To NIE jest wybór "na zawsze"**
>
> Wiele dojrzałych zespołów zaczyna od Kimballa (szybkie zwycięstwa), a z czasem dokłada elementy podejścia Inmona (wspólna warstwa `staging`/`intermediate`, ścisłe definicje kluczowych pojęć biznesowych) w miarę jak rośnie liczba zespołów korzystających z tych samych danych -- dokładnie tak, jak warstwy bronze/silver/gold z Modułu 11 kursu Databricks stopniowo się ustandaryzowały.

<a id="sec2"></a>
## 2. Fakty i wymiary

Modelowanie wymiarowe dzieli każdą tabelę na jeden z dwóch typów:

- **Tabela faktów** (*fact table*) -- rejestruje ZDARZENIA/TRANSAKCJE: każdy wiersz to
  jedno zdarzenie (np. jedna pozycja zamówienia), z LICZBOWYMI **miarami**
  (*measures* -- ilość, cena, wartość) i KLUCZAMI OBCYMI do wymiarów. Tabele faktów są
  zwykle OGROMNE (miliony/miliardy wierszy) i rosną w czasie.
- **Tabela wymiaru** (*dimension table*) -- opisuje KONTEKST zdarzenia: KIM jest klient,
  CZYM jest produkt, KIEDY (jaki dzień/miesiąc/kwartał) miało miejsce zdarzenie. Wymiary
  są zwykle MAŁE (tysiące/miliony wierszy) i zmieniają się wolno.

Kluczowe pojęcie: **granularność** (*grain*) tabeli faktów -- CO DOKŁADNIE reprezentuje
JEDEN wiersz. "Jedna pozycja zamówienia" to inna granularność niż "jedno zamówienie"
(zamówienie może mieć wiele pozycji) -- ZAWSZE ustalaj granularność PRZED napisaniem
pierwszego SQL-a, bo zmiana jej później oznacza przebudowę całego modelu.

> ⚠️ **Nigdy nie mieszaj dwóch granularności w jednej tabeli faktów**
>
> Częsty błąd: tabela faktów, w której część wierszy to "pozycje zamówień", a część to "całe zamówienia" (np. dodane osobno dla zamówień bez pozycji). SUMY policzone na takiej tabeli będą PODWÓJNIE LICZYĆ albo pomijać dane -- każda tabela faktów musi mieć DOKŁADNIE JEDNĄ, jasno zdefiniowaną granularność.

<a id="sec3"></a>
## 3. Schemat gwiazdy kontra płatek śniegu

**Schemat gwiazdy** (*star schema*) -- jedna tabela faktów w ŚRODKU, otoczona
wymiarami połączonymi z nią BEZPOŚREDNIO (jeden JOIN od faktu do każdego wymiaru).
Wymiary są ZDENORMALIZOWANE -- np. `dim_produkty` zawiera KOLUMNĘ `kategoria` wprost,
zamiast osobnej tabeli `kategorie` połączonej kolejnym kluczem.

**Płatek śniegu** (*snowflake schema*) -- wymiary są dalej ZNORMALIZOWANE na mniejsze
tabele (np. `dim_produkty` → `dim_kategorie` → `dim_dzialy`), co oznacza WIĘCEJ joinów
przy każdym zapytaniu.

| | Gwiazda | Płatek śniegu |
|---|---|---|
| Joiny w typowym zapytaniu | Mniej (szybsze zapytania) | Więcej |
| Duplikacja danych w wymiarach | Więcej (ale wymiary są MAŁE, więc to nieistotne) | Mniej |
| Czytelność dla analityka BI | Bardzo wysoka (prosty schemat) | Niższa |
| Zalecenie w nowoczesnym stosie | **Domyślny wybór** | Rzadko, gdy wymiar naprawdę ogromny |

W nowoczesnych hurtowniach kolumnowych (Snowflake, BigQuery, Databricks) koszt
przechowywania zduplikowanych danych w zdenormalizowanych wymiarach jest znikomy, a
zysk w postaci PROSTSZYCH i SZYBSZYCH zapytań ogromny -- dlatego **schemat gwiazdy jest
domyślnym wyborem** w większości nowych projektów.

<a id="sec4"></a>
## 4. Typy tabel faktów

Trzy klasyczne wzorce (Kimball nazywa je *fact table types*):

- **Transakcyjne** (*transaction fact*) -- jeden wiersz per zdarzenie biznesowe (nasza
  "pozycja zamówienia" z sekcji 5). Najczęstszy typ.
- **Migawkowe** (*periodic snapshot*) -- jeden wiersz per encja PER OKRES (np. "stan
  konta klienta na koniec KAŻDEGO miesiąca") -- używane, gdy interesuje Cię TREND stanu
  w czasie, nie pojedyncze zdarzenia.
- **Skumulowane** (*accumulating snapshot*) -- jeden wiersz per PROCES, aktualizowany w
  miarę jak proces przechodzi kolejne etapy (np. "zamówienie" z kolumnami
  `data_zlozenia`, `data_wyslania`, `data_dostarczenia`, wypełnianymi stopniowo) --
  przydatne do mierzenia czasu trwania procesów (np. "ile dni od złożenia do wysyłki").

<a id="sec5"></a>
## 5. Budujemy schemat gwiazdy

Przekształcamy nasz znany zestaw danych w schemat gwiazdy: DWA wymiary opisowe
(`dim_klienci`, `dim_produkty`), JEDEN wymiar czasu (`dim_data` -- klasyczna technika
Kimball) i JEDNĄ tabelę faktów o granularności "jedna pozycja zamówienia"
(`fact_sprzedaz`).

In [ ]:
model_kod = '''
select
    klient_id,
    upper(left(miasto, 1)) || lower(substr(miasto, 2)) as miasto,
    coalesce(segment, 'Standardowy') as segment,
    dni_od_rejestracji
from {{ source('raw_dane', 'klienci') }}
'''
with open(f"{DBT_DIR}/models/dim_klienci.sql", "w") as f:
    f.write(model_kod)

model_kod = '''
select
    produkt_id,
    kategoria,
    cena as cena_katalogowa
from {{ source('raw_dane', 'produkty') }}
'''
with open(f"{DBT_DIR}/models/dim_produkty.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "dim_klienci dim_produkty")


> ⚡ **Wymiar czasu to JEDYNY wymiar, który budujesz "z powietrza"**
>
> `dim_data` (*date dimension*) to klasyczna technika Kimball -- zamiast liczyć rok/miesiąc/dzień tygodnia w KAŻDYM zapytaniu analitycznym, budujesz JEDNĄ tabelę ze WSZYSTKIMI datami z rozsądnego zakresu (np. 5-10 lat), z gotowymi kolumnami (`rok`, `kwartal`, `nazwa_dnia_tygodnia`, `czy_weekend`...). Zapytania łączą się z nią zwykłym JOIN-em zamiast obliczać te wartości za każdym razem.

In [ ]:
model_kod = '''
select
    data::date as data_id,
    extract(year from data) as rok,
    extract(quarter from data) as kwartal,
    extract(month from data) as miesiac,
    extract(day from data) as dzien_miesiaca,
    extract(dow from data) as dzien_tygodnia,
    case when extract(dow from data) in (0, 6) then true else false end as czy_weekend
from generate_series(DATE '2023-01-01', DATE '2026-12-31', INTERVAL 1 DAY) as t(data)
'''
with open(f"{DBT_DIR}/models/dim_data.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "dim_data")


In [ ]:
# fact_sprzedaz -- granularnosc: JEDNA POZYCJA ZAMOWIENIA (nie jedno zamowienie!)
model_kod = '''
select
    poz.pozycja_id,
    z.zamowienie_id,
    z.klient_id,
    poz.produkt_id,
    z.data_zamowienia::date as data_id,
    poz.ilosc,
    poz.cena_jednostkowa,
    poz.rabat,
    round(poz.ilosc * poz.cena_jednostkowa * (1 - poz.rabat), 2) as wartosc_pozycji,
    z.status
from {{ source('raw_dane', 'pozycje_zamowien') }} poz
inner join {{ source('raw_dane', 'zamowienia') }} z on poz.zamowienie_id = z.zamowienie_id
'''
with open(f"{DBT_DIR}/models/fact_sprzedaz.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "fact_sprzedaz")


In [ ]:
# Typowe zapytanie analityczne na gotowym schemacie gwiazdy -- JEDEN join na wymiar
zapytanie = '''
select
    dd.rok,
    dd.kwartal,
    dk.segment,
    dp.kategoria,
    round(sum(fs.wartosc_pozycji), 2) as przychod
from main.fact_sprzedaz fs
inner join main.dim_klienci dk on fs.klient_id = dk.klient_id
inner join main.dim_produkty dp on fs.produkt_id = dp.produkt_id
inner join main.dim_data dd on fs.data_id = dd.data_id
where fs.status != 'Anulowane'
group by dd.rok, dd.kwartal, dk.segment, dp.kategoria
order by przychod desc
limit 10
'''
con = duckdb.connect(DUCKDB_PATH)
print(con.execute(zapytanie).fetchdf())
con.close()


> 🏭 **To zapytanie mogliby napisać BI/analitycy SAMI**
>
> Zauważ, jak CZYTELNE jest to zapytanie: cztery proste JOIN-y (fakt -> każdy wymiar), zrozumiałe nazwy kolumn, żadnej logiki biznesowej do odgadywania -- to właśnie cel modelowania wymiarowego. Analityk pracujący w Tableau/Power BI (albo w SQL-u z kursu SQL) może budować dowolne przekroje BEZ pomocy inżyniera danych, bo cała trudna praca (czyszczenie, łączenie, definicje) już się wydarzyła w warstwie gwiazdy.

<a id="sec6"></a>
## 6. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- Kimball (bottom-up, szybkie zwycięstwa) kontra Inmon (top-down, gwarantowana spójność),
- fakty (zdarzenia + miary) i wymiary (kontekst opisowy) oraz pojęcie granularności,
- schemat gwiazdy (domyślny wybór) kontra płatek śniegu,
- trzy typy tabel faktów: transakcyjne, migawkowe, skumulowane,
- praktyczny schemat gwiazdy: `dim_klienci`, `dim_produkty`, `dim_data`,
  `fact_sprzedaz` (granularność: jedna pozycja zamówienia).

### 📝 Ćwiczenie 1: Jaka granularność?

Dla każdej z poniższych tabel faktów określ granularność JEDNYM zdaniem: (a) tabela z jednym wierszem per klient per miesiąc, pokazująca sumę wydatków w tym miesiącu, (b) tabela z jednym wierszem per kliknięcie w reklamę, (c) tabela z jednym wierszem per zamówienie (nie per pozycja), z kolumną `liczba_pozycji` i `wartosc_calkowita`.

<details>
<summary>Pokaż rozwiązanie</summary>

```text
# (a) Granularnosc: jeden klient w jednym miesiacu (fakt migawkowy/periodic snapshot).
# (b) Granularnosc: jedno klikniecie w reklame (fakt transakcyjny).
# (c) Granularnosc: jedno zamowienie (agregacja z poziomu pozycji do poziomu zamowienia
#     -- inna granularnosc niz nasze fact_sprzedaz z sekcji 5!).
print("Zobacz komentarze powyzej.")
```

</details>

### 📝 Ćwiczenie 2: Dodaj wymiar pracowników

Zbuduj `dim_pracownicy.sql` na podstawie danych, które wygenerujesz sam (kolumny: `pracownik_id`, `region`, `dzial` -- 12 pracowników, regiony `["Polnoc","Poludnie","Wschod","Zachod"]`, działy `["Sprzedaz","Obsluga"]`) i załaduj ją jako SEED (Moduł 5), a nie jako model -- uzasadnij DLACZEGO seed jest tu rozsądnym wyborem.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
import pandas as pd

pracownicy = pd.DataFrame({
    "pracownik_id": range(1, 13),
    "region": (["Polnoc","Poludnie","Wschod","Zachod"] * 3)[:12],
    "dzial": (["Sprzedaz","Obsluga"] * 6)[:12],
})
pracownicy.to_csv(f"{DBT_DIR}/seeds/dim_pracownicy.csv", index=False)

dbt("seed", "--select", "dim_pracownicy")

# Seed jest rozsadnym wyborem, bo: dane sa MALE (12 wierszy), RZADKO sie zmieniaja
# (nowi pracownicy to rzadkosc), i nie pochodza z zadnego zewnetrznego systemu
# zrodlowego -- to raczej "dane referencyjne" zarzadzane przez zespol danych,
# dokladnie jak mapa_regionow.csv z Modulu 5.
print("Zobacz uzasadnienie w komentarzu powyzej.")
```

</details>

### 📝 Ćwiczenie 3: Zapytanie na schemacie gwiazdy

Napisz zapytanie SQL na gotowym schemacie gwiazdy z sekcji 5, które zwraca sumę `wartosc_pozycji` PER `miasto` klienta, TYLKO dla zamówień z 2024 roku, posortowaną malejąco.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
zapytanie = '''
select
    dk.miasto,
    round(sum(fs.wartosc_pozycji), 2) as przychod
from main.fact_sprzedaz fs
inner join main.dim_klienci dk on fs.klient_id = dk.klient_id
inner join main.dim_data dd on fs.data_id = dd.data_id
where dd.rok = 2024 and fs.status != 'Anulowane'
group by dk.miasto
order by przychod desc
'''
con = duckdb.connect(DUCKDB_PATH)
print(con.execute(zapytanie).fetchdf())
con.close()
```

</details>

> 🔥 **Wyzwanie: fakt skumulowany dla cyklu życia zamówienia**
>
> Zbuduj model `fact_cykl_zamowienia.sql` typu SKUMULOWANEGO (accumulating snapshot) z granularnością "jedno zamówienie": kolumny `zamowienie_id`, `klient_id`, `data_zamowienia`, oraz WYLICZONA kolumna `dni_do_dzisiaj` (liczba dni od `data_zamowienia` do dzisiaj, użyj `current_date - data_zamowienia`) jako prosty przybliżony wskaźnik "wieku" zamówienia (w prawdziwym potoku miałbyś osobne kolumny `data_wyslania`/`data_dostarczenia`, aktualizowane w miarę postępu procesu).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
model_kod = '''
select
    zamowienie_id,
    klient_id,
    data_zamowienia::date as data_zamowienia,
    (current_date - data_zamowienia::date) as dni_do_dzisiaj,
    status
from {{ source('raw_dane', 'zamowienia') }}
'''
with open(f"{DBT_DIR}/models/fact_cykl_zamowienia.sql", "w") as f:
    f.write(model_kod)

dbt("run", "--select", "fact_cykl_zamowienia")

con = duckdb.connect(DUCKDB_PATH)
print(con.execute("SELECT * FROM main.fact_cykl_zamowienia ORDER BY dni_do_dzisiaj LIMIT 5").fetchdf())
con.close()
```

Granularnosc tego faktu (jedno zamowienie) ROZNI SIE od fact_sprzedaz (jedna pozycja) -- to normalne i oczekiwane: rozne pytania biznesowe czesto wymagaja faktow o roznej granularnosci, nawet nad tymi samymi danymi zrodlowymi.

</details>

## Co dalej?

W **Module 7** zajmiemy się problemem, którego schemat gwiazdy z tego modułu CELOWO
ignorował: co się dzieje, gdy atrybut wymiaru SIĘ ZMIENIA w czasie (klient zmienia
miasto, produkt zmienia kategorię)? To domena **Slowly Changing Dimensions (SCD)**.